# ShowPDF4D — Interactive PDF Analysis (Experimental SiN)

Interactive PDF analysis of an experimental 4D-STEM nanodiffraction dataset of amorphous SiN,
acquired on a TitanX at 200 kV with an Arina detector (512×512 scan, 192×192 detector).

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import quantem as em
from quantem.diffraction import PairDistributionFunction
from quantem.widget import ShowPDF4D, IO
from quantem.widget.detector import virtual_images
import numpy as np

/Users/Karen/Stanford/repos/quantem/.venv/lib/python3.12/site-packages/anywidget/_util.py:283: UserWarning: anywidget: Live-reloading feature is disabled. To enable, please install the 'watchfiles' package.
  start_thread=_should_start_thread(path),


## Load Arina 4D-STEM data

In [3]:
path = "/Volumes/KME_SSD/SiN/"
file = "SiN_w_AuNP_005_master.h5"
result = IO.arina_file(path + file, det_bin=2)
print(result)

from quantem.core.datastructures import Dataset4dstem
ds = Dataset4dstem.from_array(array=result.data)
print(f"4D shape: {ds.array.shape}")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Volumes/KME_SSD/SiN/SiN_w_AuNP_005_master.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# # Mask hot pixels
# mask_hot = ds.dp_mean.array > 1e4
# ds.median_filter_masked_pixels(mask_hot)
# print(f"Masked {mask_hot.sum()} hot pixels")

## Polar transform (center of detector, no origin search)

In [ ]:
# Skip origin finding — use detector center
ny, nx = ds.array.shape[2], ds.array.shape[3]
polar = ds.polar_transform(
    origin_array=[(ny - 1) / 2.0, (nx - 1) / 2.0],
)
polar.sampling[3] = 0.052  # 0.026 * 2 for det_bin=2
print(f"Polar shape: {polar.shape}")
print(f"q range: 0 to {polar.shape[3] * polar.sampling[3]:.2f} Å⁻¹")

## Build PDF and launch widget

In [ ]:
pdf = PairDistributionFunction.from_data(polar)
pdf.polar.sampling[3] = 0.052  # 0.026 * 2 for det_bin=2
print(f"q range: {pdf.qq[0]:.4f} to {pdf.qq[-1]:.4f} Å⁻¹")

In [ ]:
# BF virtual image for navigation panel
bf, adf, haadf = virtual_images(ds.array)
print(f"BF shape: {bf.shape}")

In [4]:
w = ShowPDF4D(
    pdf,
    nav_image=bf,
    title="SiN Experimental PDF",
    k_min_fit=0.05,
    k_max_fit=2.2,
    k_min_window=0.35,
    k_max_window=1.9,
    r_max=10.0,
)
w

NameError: name 'pdf' is not defined

In [9]:
w.summary()

SiN Experimental PDF
════════════════════════════════
Scan:     512 × 512
k range:  [0.00, 2.44] Å⁻¹
Fit:      k=[0.05, 2.20]
Output:   r=[0.00, 10.00], step=0.02
Plot:     Gr
Mask:     full scan (no mask)
